# Suspicious URL Detection and Website Risk Analysis
**Goal:** classify a URL as *benign* or *suspicious/phishing* using machine learning on lexical features, then combine the model with rule-based indicators (and optional page-content signals) into a human-readable **risk score (0-100)** with Low / Medium / High levels.

**Pipeline:** Data → Feature extraction → EDA → Model training & comparison → Evaluation → Risk scoring engine → Live URL analyzer

> Run all cells top to bottom (Runtime → Run all). No API keys are needed.

## 1. Setup

In [ ]:
import re, math, warnings, random, socket
from urllib.parse import urlparse
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_style('whitegrid')
print('Setup complete')

## 2. Dataset
Two options:
* **Option A (recommended for real results):** upload a CSV with columns `url` and `label` (`1` = phishing/malicious, `0` = benign). Public sources: PhishTank, OpenPhish, URLhaus (malicious) and Tranco top-sites list (benign). Set `CSV_PATH` below.
* **Option B (default):** a built-in generator that produces a realistic **synthetic** dataset (including hard/ambiguous cases) so the notebook runs anywhere.

> Metrics on synthetic data demonstrate the pipeline only; retrain on real data before drawing real-world conclusions.

In [ ]:
CSV_PATH = None   # e.g. '/content/urls.csv' after uploading to Colab

LEGIT_DOMAINS = ['google.com','youtube.com','wikipedia.org','amazon.com','github.com','microsoft.com','apple.com',
 'bbc.co.uk','nytimes.com','stackoverflow.com','reddit.com','linkedin.com','netflix.com','paypal.com','ebay.com',
 'adobe.com','dropbox.com','nasa.gov','mit.edu','harvard.edu','irs.gov','who.int','cloudflare.com','python.org',
 'medium.com','imdb.com','spotify.com','zoom.us','walmart.com','bankofamerica.com','chase.com','hdfcbank.com',
 'sbi.co.in','flipkart.com','swiggy.com','coursera.org','kaggle.com','twitter.com','instagram.com','whatsapp.com']
BRANDS = ['paypal','amazon','apple','google','microsoft','netflix','facebook','sbi','hdfc','icici','chase','dhl','fedex','instagram']
BAD_TLDS = ['xyz','top','tk','ml','ga','cf','gq','click','link','work','support','icu','buzz','live','online','site']
KEYWORDS = ['login','secure','verify','account','update','confirm','banking','signin','password','wallet','billing','suspend','unlock','recover']
WORDS = ['news','shop','blog','docs','help','about','products','careers','contact','search','video','watch','post','article','store','events','pricing','support','learn','home']
rand_str = lambda n: ''.join(random.choices('abcdefghijklmnopqrstuvwxyz0123456789', k=n))

def gen_legit():
    d = random.choice(LEGIT_DOMAINS)
    sub = random.choice(['', '', 'www.', 'www.', 'mail.', 'accounts.', 'support.', 'docs.'])
    scheme = 'https://' if random.random() < 0.93 else 'http://'
    depth = random.choice([0,0,1,1,2,3])
    path = '/' + '/'.join(random.choice(WORDS) + (str(random.randint(1,9999)) if random.random()<0.2 else '') for _ in range(depth)) if depth else ''
    q = ''
    if random.random() < 0.25: q = '?id=%d&ref=%s' % (random.randint(1,99999), random.choice(WORDS))
    if random.random() < 0.05: path += '/login'          # legit pages also contain keywords (hard case)
    return scheme + sub + d + path + q

def gen_phish():
    kind = random.random()
    scheme = 'http://' if random.random() < 0.55 else 'https://'
    brand, kw = random.choice(BRANDS), random.choice(KEYWORDS)
    if kind < 0.18:       # IP-based host
        host = '.'.join(str(random.randint(1,254)) for _ in range(4))
        path = '/' + kw + '/' + rand_str(random.randint(4,10)) + '.php'
    elif kind < 0.42:     # brand impersonation in subdomain
        host = '%s-%s.%s.%s' % (brand, kw, rand_str(random.randint(5,10)), random.choice(BAD_TLDS))
        path = '/' + rand_str(random.randint(3,8)) + '/' + kw
    elif kind < 0.62:     # long subdomain chain on free host
        host = '%s.%s.%s.%s' % (brand, kw, rand_str(6), random.choice(['000webhostapp.com','weebly.com','blogspot.com','web.app']))
        path = '/' + kw + '.html'
    elif kind < 0.75:     # @ trick / redirect
        host = random.choice(LEGIT_DOMAINS)
        scheme = 'http://'
        return scheme + host + '@' + rand_str(8) + '.' + random.choice(BAD_TLDS) + '/' + kw
    elif kind < 0.88:     # typosquat
        b = brand.replace('o','0').replace('l','1') if random.random()<0.5 else brand + brand[-1]
        host = b + '.' + random.choice(['com','net','co','info'] + BAD_TLDS)
        path = '/' + random.choice(['','signin','account/verify','secure'])
    else:                 # hard case: clean-looking short URL
        host = rand_str(random.randint(6,11)) + '.' + random.choice(['com','net','org','xyz'])
        path = random.choice(['', '/index.php', '/' + rand_str(5)])
    q = ('?' + 'session=' + rand_str(16) + '&email=') if random.random() < 0.3 else ''
    return scheme + host + path + q

def build_dataset(n=6000):
    rows  = [(gen_legit(), 0) for _ in range(n//2)] + [(gen_phish(), 1) for _ in range(n//2)]
    df = pd.DataFrame(rows, columns=['url','label']).drop_duplicates('url')
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

if CSV_PATH:
    raw = pd.read_csv(CSV_PATH)[['url','label']].dropna().drop_duplicates('url')
    data_source = 'real CSV'
else:
    raw = build_dataset(8000); data_source = 'synthetic'
print(data_source, raw.shape); raw.head(8)

## 3. Feature Engineering
Each URL is converted into numeric features that capture known phishing traits:
length and structure, special characters, IP-as-host, `@` symbol, subdomain depth, entropy (random-looking strings), suspicious keywords, brand names outside the registered domain, risky TLDs, URL shorteners, and HTTPS usage.

In [ ]:
SHORTENERS = ['bit.ly','tinyurl.com','goo.gl','t.co','ow.ly','is.gd','buff.ly','rebrand.ly','cutt.ly']
IP_RE = re.compile(r'^(\d{1,3}\.){3}\d{1,3}$')
MULTI_TLD = {'co.uk','co.in','com.au','org.uk','ac.uk','gov.uk'}

def entropy(s):
    if not s: return 0.0
    c = Counter(s); n = len(s)
    return -sum(v/n*math.log2(v/n) for v in c.values())

def split_host(host):
    parts = host.split('.')
    if len(parts) >= 3 and '.'.join(parts[-2:]) in MULTI_TLD:
        return '.'.join(parts[:-3]), parts[-3], '.'.join(parts[-2:])
    if len(parts) >= 2:
        return '.'.join(parts[:-2]), parts[-2], parts[-1]
    return '', host, ''

def extract_features(url):
    u = url.strip()
    p = urlparse(u if '://' in u else 'http://' + u)
    host = (p.hostname or '').lower()
    sub, domain, tld = split_host(host)
    path, query = p.path or '', p.query or ''
    low = u.lower()
    return {
        'url_len': len(u), 'host_len': len(host), 'path_len': len(path), 'query_len': len(query),
        'n_dots': u.count('.'), 'n_hyphens_host': host.count('-'), 'n_at': u.count('@'),
        'n_digits': sum(ch.isdigit() for ch in u), 'digit_ratio': sum(ch.isdigit() for ch in u)/max(len(u),1),
        'n_special': sum(not ch.isalnum() for ch in u), 'n_slash': u.count('/'),
        'n_params': query.count('&') + (1 if query else 0), 'path_depth': len([x for x in path.split('/') if x]),
        'has_ip': int(bool(IP_RE.match(host))), 'is_https': int(p.scheme == 'https'),
        'n_subdomains': len([x for x in sub.split('.') if x]),
        'entropy_host': entropy(host), 'entropy_path': entropy(path + query),
        'n_keywords': sum(k in low for k in KEYWORDS),
        'brand_in_sub_or_path': int(any(b in (sub + path).lower() for b in BRANDS) and not any(b == domain for b in BRANDS)),
        'brand_in_domain_not_exact': int(any(b in domain and b != domain for b in BRANDS)),
        'bad_tld': int(tld in BAD_TLDS), 'is_shortener': int(host in SHORTENERS),
        'has_port': int(p.port is not None), 'double_slash_path': int('//' in path),
        'long_random_token': int(bool(re.search(r'[a-z0-9]{16,}', low))),
        'has_php_html': int(bool(re.search(r'\.(php|html?|asp)x?$', path))),
        'free_host': int(any(h in host for h in ['000webhostapp','weebly','blogspot','web.app','herokuapp','netlify'])),
    }

X = pd.DataFrame([extract_features(u) for u in raw.url]); y = raw.label.values
print('Features:', X.shape[1]); X.head()

## 4. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
raw.label.map({0:'Benign',1:'Phishing'}).value_counts().plot.bar(ax=ax[0], color=['#4C9F70','#D1495B'], rot=0); ax[0].set_title('Class balance')
sns.boxplot(x=y, y=X.url_len, ax=ax[1], palette=['#4C9F70','#D1495B']); ax[1].set_title('URL length by class'); ax[1].set_xticklabels(['Benign','Phishing'])
sns.boxplot(x=y, y=X.entropy_host, ax=ax[2], palette=['#4C9F70','#D1495B']); ax[2].set_title('Host entropy by class'); ax[2].set_xticklabels(['Benign','Phishing'])
plt.tight_layout(); plt.savefig('eda1.png', dpi=130); plt.show()

corr = X.assign(label=y).corr()['label'].drop('label').sort_values()
plt.figure(figsize=(7,8)); corr.plot.barh(color=np.where(corr>0,'#D1495B','#4C9F70'))
plt.title('Feature correlation with phishing label'); plt.tight_layout(); plt.savefig('eda2.png', dpi=130); plt.show()

## 5. Train / Test Split and Model Comparison

In [ ]:
X_tr, X_te, y_tr, y_te, url_tr, url_te = train_test_split(X, y, raw.url, test_size=0.2, stratify=y, random_state=SEED)
models = {
 'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
 'Random Forest':       RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
 'Gradient Boosting':   GradientBoostingClassifier(random_state=SEED),
}
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
res, fitted = [], {}
for name, m in models.items():
    cv = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='f1').mean()
    m.fit(X_tr, y_tr); fitted[name] = m
    pr, pb = m.predict(X_te), m.predict_proba(X_te)[:,1]
    res.append({'Model':name, 'CV F1':cv, 'Accuracy':accuracy_score(y_te,pr), 'Precision':precision_score(y_te,pr),
                'Recall':recall_score(y_te,pr), 'F1':f1_score(y_te,pr), 'ROC-AUC':roc_auc_score(y_te,pb)})
results = pd.DataFrame(res).set_index('Model').round(4); results

## 6. Evaluation (best model)

In [ ]:
best_name = results['F1'].idxmax(); best = fitted[best_name]; print('Best model:', best_name)
pred, proba = best.predict(X_te), best.predict_proba(X_te)[:,1]
print(classification_report(y_te, pred, target_names=['Benign','Phishing'], digits=4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(confusion_matrix(y_te, pred), annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing']); ax[0].set_title('Confusion matrix - '+best_name)
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
for n, m in fitted.items():
    fpr, tpr, _ = roc_curve(y_te, m.predict_proba(X_te)[:,1]); ax[1].plot(fpr, tpr, label='%s (AUC=%.3f)' % (n, roc_auc_score(y_te, m.predict_proba(X_te)[:,1])))
ax[1].plot([0,1],[0,1],'k--'); ax[1].set_title('ROC curves'); ax[1].set_xlabel('False positive rate'); ax[1].set_ylabel('True positive rate'); ax[1].legend()
plt.tight_layout(); plt.savefig('eval.png', dpi=130); plt.show()

rf = fitted['Random Forest']
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(15)
plt.figure(figsize=(7,5)); imp.plot.barh(color='#2E5EAA'); plt.title('Top 15 feature importances (Random Forest)')
plt.tight_layout(); plt.savefig('importance.png', dpi=130); plt.show()

err = pd.DataFrame({'url':url_te.values,'actual':y_te,'pred':pred,'p':proba.round(3)})
print('Misclassified examples:'); err[err.actual != err.pred].head(10)

## 7. Website Risk Analysis Engine
A single ML probability is not enough for a security analyst. The engine below blends:
1. **ML probability** (weight 60%),
2. **Rule-based indicators** (weight 40%) with explicit reasons, and
3. optional **page-content signals** (password form on HTTP, hidden iframes, many external links, mismatched form action) and **DNS resolution**.

Output: score 0-100, level (Low < 35, Medium 35-65, High > 65) and a list of human-readable reasons.

In [ ]:
import requests
from bs4 import BeautifulSoup

RULES = [  # (feature condition, points, explanation)
 (lambda f: f['has_ip'],                 25, 'Host is a raw IP address'),
 (lambda f: f['n_at'] > 0,               20, "URL contains '@' (can hide the real destination)"),
 (lambda f: f['bad_tld'],                12, 'Top-level domain commonly abused'),
 (lambda f: f['brand_in_sub_or_path'],   20, 'Brand name appears outside the registered domain (impersonation)'),
 (lambda f: f['brand_in_domain_not_exact'], 15, 'Domain looks like a brand typo/variant'),
 (lambda f: f['n_keywords'] >= 2,        12, 'Multiple sensitive keywords (login/verify/secure...)'),
 (lambda f: f['n_subdomains'] >= 3,      10, 'Unusually deep subdomain chain'),
 (lambda f: not f['is_https'],            8, 'No HTTPS'),
 (lambda f: f['url_len'] > 90,            8, 'Very long URL'),
 (lambda f: f['entropy_host'] > 3.6,      8, 'Random-looking hostname'),
 (lambda f: f['is_shortener'],           10, 'URL shortener hides destination'),
 (lambda f: f['free_host'],              10, 'Hosted on a free website builder'),
 (lambda f: f['long_random_token'],       6, 'Long random token in URL'),
]

def content_signals(url, timeout=6):
    out = {'fetched': False, 'points': 0, 'reasons': []}
    try:
        r = requests.get(url, timeout=timeout, headers={'User-Agent':'Mozilla/5.0'}, allow_redirects=True)
        soup = BeautifulSoup(r.text[:300000], 'html.parser'); out['fetched'] = True
        host = urlparse(r.url).hostname or ''
        if len(r.history) >= 3: out['points'] += 8; out['reasons'].append('Multiple redirects (%d)' % len(r.history))
        if soup.find('input', {'type':'password'}):
            out['points'] += 10; out['reasons'].append('Page contains a password field')
            if urlparse(r.url).scheme != 'https': out['points'] += 15; out['reasons'].append('Password field served over HTTP')
        for fm in soup.find_all('form'):
            act = fm.get('action', '')
            if act.startswith('http') and (urlparse(act).hostname or host) != host:
                out['points'] += 15; out['reasons'].append('Form submits to a different domain'); break
        if soup.find('iframe', style=re.compile('display:\\s*none|visibility:\\s*hidden', re.I)):
            out['points'] += 10; out['reasons'].append('Hidden iframe')
        links = [a.get('href','') for a in soup.find_all('a', href=True)]
        ext = [l for l in links if l.startswith('http') and (urlparse(l).hostname or host) != host]
        if links and len(ext)/len(links) > 0.8 and len(links) > 5: out['points'] += 8; out['reasons'].append('Most links point to other domains')
    except Exception as e:
        out['reasons'].append('Content not fetched (%s)' % type(e).__name__)
    return out

def analyze_url(url, fetch_content=False, model=None):
    model = model or best
    f = extract_features(url)
    p_ml = float(model.predict_proba(pd.DataFrame([f]))[0,1])
    hits = [(pts, why) for cond, pts, why in RULES if cond(f)]
    rule_score = min(sum(p for p,_ in hits), 100)
    reasons = [w for _,w in sorted(hits, reverse=True)]
    score = 0.6*p_ml*100 + 0.4*rule_score
    host = urlparse(url if '://' in url else 'http://'+url).hostname or ''
    try: socket.gethostbyname(host); dns_ok = True
    except Exception: dns_ok = False
    if not dns_ok: reasons.append('Domain does not resolve (DNS)')
    if fetch_content:
        c = content_signals(url if '://' in url else 'http://'+url); score = min(100, score + c['points']); reasons += c['reasons']
    score = round(min(score,100),1)
    level = 'LOW' if score < 35 else ('MEDIUM' if score <= 65 else 'HIGH')
    return {'url':url,'ml_probability':round(p_ml,3),'rule_score':rule_score,'risk_score':score,'risk_level':level,'reasons':reasons or ['No suspicious indicators']}

def show(r):
    icon = {'LOW':'LOW ', 'MEDIUM':'MED ', 'HIGH':'HIGH'}[r['risk_level']]
    print('[%s] %5.1f/100  ML p=%.2f  %s' % (icon, r['risk_score'], r['ml_probability'], r['url']))
    for w in r['reasons'][:5]: print('        -', w)

## 8. Demo on hand-written URLs

In [ ]:
tests = ['https://www.wikipedia.org/wiki/Phishing',
 'https://github.com/features',
 'http://paypal-secure-login.verify-account.xyz/signin',
 'http://192.168.44.12/login/update.php?session=ab12cd34ef56gh78',
 'http://www.google.com@malicious-site.top/verify',
 'http://amaz0n.com/account/verify',
 'https://accounts.google.com/signin',
 'https://netflix.billing.update.a8f3k2.000webhostapp.com/secure.html']
demo = [analyze_url(t) for t in tests]
for r in demo: show(r)
pd.DataFrame(demo)[['url','ml_probability','rule_score','risk_score','risk_level']]

## 9. Risk-score validation on the held-out test set

In [ ]:
sc = np.array([analyze_url(u)['risk_score'] for u in url_te.values[:600]]); yt = y_te[:600]
lv = pd.cut(sc, [-1,34.99,65,101], labels=['LOW','MEDIUM','HIGH'])
tab = pd.crosstab(lv, pd.Series(yt).map({0:'Benign',1:'Phishing'}).values); print(tab)
print('ROC-AUC of combined risk score: %.4f' % roc_auc_score(yt, sc))
plt.figure(figsize=(7,4)); plt.hist(sc[yt==0], bins=30, alpha=.7, label='Benign', color='#4C9F70'); plt.hist(sc[yt==1], bins=30, alpha=.7, label='Phishing', color='#D1495B')
plt.axvline(35,ls='--',c='k'); plt.axvline(65,ls='--',c='k'); plt.legend(); plt.title('Risk-score distribution'); plt.xlabel('Risk score'); plt.tight_layout(); plt.savefig('riskdist.png', dpi=130); plt.show()

## 10. Try your own URL
Set `fetch_content=True` to also download the page (only analyse sites you are allowed to visit; never open unknown links in a normal browser).

In [ ]:
user_url = input('Enter a URL to analyze: ') if False else 'http://secure-paypal-login.example-verify.top/account'
show(analyze_url(user_url, fetch_content=False))

## 11. Save the model

In [ ]:
import joblib
joblib.dump({'model':best,'features':list(X.columns)}, 'url_risk_model.joblib'); print('saved url_risk_model.joblib')

## 12. Limitations & Future Work
* Lexical features alone can be evaded by attackers using clean-looking domains; add WHOIS domain age, TLS certificate details, and page-content/visual (screenshot) features.
* Synthetic data overstates accuracy; retrain on PhishTank/OpenPhish + Tranco data and evaluate with a time-based split.
* Add threat-intel lookups (Google Safe Browsing, VirusTotal) and deploy as a Flask/Streamlit app or browser extension.